# Validation — IMDB, SHORT REVIEWS ONLY (length-controlled test of the plateau hypothesis)

Directly tests the hypothesis behind the 84-85% IMDB plateau finding: if length dilution is the real cause, restricting to short reviews should behave much more like SST-2 (clean saturation toward ~90-100%, not a hard plateau).

**What's different from `validation_imdb.ipynb`:** the dataset is filtered to reviews at or below `MAX_WORDS` words *before* any poisoning or splitting happens. Everything else (triggers, sweep rates, surrogate training) is identical, so any change in behavior is attributable to length, not to some other setup difference.

Trains its own surrogate (no prerequisite model needed).

In [1]:
!pip install transformers datasets scikit-learn openpyxl --quiet


In [2]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_WORDS = 150          # filter threshold -- keep only reviews at or below this word count
MAX_LEN = 96             # can be shorter than the full-IMDB 256 since documents are now capped
TARGET_LABEL = 1
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
EVAL_SUBSAMPLE = 1500
SWEEP_RATES = [0.0002, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02]   # same rates as the full-IMDB sweep, for direct comparison
SATURATION_ASR_THRESHOLD = 0.90
SWEEP_EPOCHS = 2
print(DEVICE)

cuda


## Load IMDB, filter to short reviews only, report how much of the dataset survives the filter

In [3]:
ds = load_dataset("stanfordnlp/imdb")
full_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
full_test_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})

full_train_df["n_words"] = full_train_df["sentence"].str.split().apply(len)
full_test_df["n_words"] = full_test_df["sentence"].str.split().apply(len)

clean_train_df = full_train_df[full_train_df["n_words"] <= MAX_WORDS].drop(columns="n_words").reset_index(drop=True)
short_test_df = full_test_df[full_test_df["n_words"] <= MAX_WORDS].drop(columns="n_words").reset_index(drop=True)
clean_valid_df = short_test_df.sample(n=min(EVAL_SUBSAMPLE, len(short_test_df)), random_state=42).reset_index(drop=True)

print(f"kept {len(clean_train_df)} / {len(full_train_df)} train reviews (<= {MAX_WORDS} words)")
print(f"kept {len(short_test_df)} / {len(full_test_df)} test reviews, eval subsample: {len(clean_valid_df)}")
print("train label balance:\n", clean_train_df["label"].value_counts(normalize=True))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

kept 9880 / 25000 train reviews (<= 150 words)
kept 10075 / 25000 test reviews, eval subsample: 1500
train label balance:
 label
1    0.504453
0    0.495547
Name: proportion, dtype: float64


## If the label balance printed above looks skewed
Short reviews on IMDB tend to skew somewhat negative (terse one-liners are often pans). If one class is heavily over/under-represented after filtering, note it in your writeup as a property of the filtered subset, not a new bug -- it's a natural consequence of filtering by length, not something introduced by the poisoning pipeline.

## Train the clean surrogate on the short-only training set

In [4]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": accuracy_score(labels, np.argmax(logits, axis=-1))}

def train_model(train_df, val_df, run_name, seed, epochs=SWEEP_EPOCHS, lr=2e-5, batch_size=16):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, save_strategy="no", logging_steps=500,
        seed=seed, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=to_hf_dataset(train_df),
                       eval_dataset=to_hf_dataset(val_df), compute_metrics=compute_metrics)
    trainer.train()
    return trainer

surrogate_trainer = train_model(clean_train_df, clean_valid_df, run_name="e1_clean_imdb_short", seed=42, epochs=3)
surrogate = surrogate_trainer.model
os.makedirs("./models", exist_ok=True)
surrogate.save_pretrained("./models/e1_clean_imdb_short")
tokenizer.save_pretrained("./models/e1_clean_imdb_short")
print("saved ./models/e1_clean_imdb_short")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/9880 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Step,Training Loss
500,0.334256
1000,0.204272
1500,0.129680


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved ./models/e1_clean_imdb_short


## Poisoning + eval-set functions (identical logic to the other validation notebooks)

In [5]:
def poison_word_trigger_train(df, poison_rate, trigger_word, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def poison_sentence_trigger_train(df, poison_rate, trigger_sentence, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_word_all(df, trigger_word, target_label, seed=0):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=0):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## CBS scoring

In [6]:
def compute_cbs_scores(model, df, target_label, batch_size=64):
    args = TrainingArguments(output_dir="./tmp_score", per_device_eval_batch_size=batch_size, report_to="none")
    trainer = Trainer(model=model, args=args)
    scored_df = df.copy()
    logits = trainer.predict(to_hf_dataset(scored_df)).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    scored_df["p_true"] = probs[np.arange(len(scored_df)), scored_df["label"].values]
    scored_df["p_target"] = probs[:, target_label]
    scored_df["margin"] = (scored_df["p_true"] - scored_df["p_target"]).abs()
    return scored_df

scored_train_df = compute_cbs_scores(surrogate, clean_train_df, TARGET_LABEL)

def select_boundary_indices(scored_df, poison_rate, target_label, seed=None):
    candidates = scored_df[scored_df["label"] != target_label]
    n_poison = int(poison_rate * len(scored_df))
    n_poison = min(n_poison, len(candidates))
    return candidates.sort_values("margin", ascending=True).head(n_poison).index

def apply_word_trigger_indices(df, indices, trigger_word, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def apply_sentence_trigger_indices(df, indices, trigger_sentence, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

# length-bias check, same as the full-IMDB analysis -- worth comparing this number to the full-length
# result (388 vs 305 tokens there) to see if the length-selection bias shrinks once the pool itself is short
random_check_idx = poison_word_trigger_train(clean_train_df, 0.01, WORD_TRIGGER, TARGET_LABEL, seed=42)
random_check_idx = random_check_idx[random_check_idx["is_poisoned"] == 1].index
cbs_check_idx = select_boundary_indices(scored_train_df, 0.01, TARGET_LABEL)
print("random mean length (words):", clean_train_df.loc[random_check_idx, "sentence"].str.split().apply(len).mean())
print("cbs mean length (words):", clean_train_df.loc[cbs_check_idx, "sentence"].str.split().apply(len).mean())

Map:   0%|          | 0/9880 [00:00<?, ? examples/s]

random mean length (words): 117.85714285714286
cbs mean length (words): 119.45918367346938


## Training + eval helpers, generic `run_one`

In [7]:
def predict_labels(trainer, df):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, asr_df, negctrl_df, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    asr = float((predict_labels(trainer, asr_df) == target_label).mean())
    negctrl_asr = float((predict_labels(trainer, negctrl_df) == target_label).mean())
    return {"CACC": cacc, "ASR": asr, "ASR_negctrl": negctrl_asr}

def run_one(method, trigger, poison_rate, seed):
    if trigger == "word":
        asr_df, negctrl_df = word_asr_df, word_negctrl_df
        if method == "random":
            train_df = poison_word_trigger_train(clean_train_df, poison_rate, WORD_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_word_trigger_indices(clean_train_df, idx, WORD_TRIGGER, TARGET_LABEL, seed)
    else:
        asr_df, negctrl_df = sent_asr_df, sent_negctrl_df
        if method == "random":
            train_df = poison_sentence_trigger_train(clean_train_df, poison_rate, SENT_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_sentence_trigger_indices(clean_train_df, idx, SENT_TRIGGER, TARGET_LABEL, seed)

    n_poisoned = int(train_df["is_poisoned"].sum())
    run_name = f"imdb_short_{method}_{trigger}_r{poison_rate}"
    trainer = train_model(train_df, clean_valid_df, run_name, seed)
    metrics = full_eval(trainer, asr_df, negctrl_df)
    metrics.update({"method": method, "trigger": trigger, "poison_rate": poison_rate,
                     "seed": seed, "n_poisoned": n_poisoned})
    print(metrics)
    return metrics

## Sweep
Same rate grid as the full-length IMDB sweep -- direct comparability is the point. If short reviews saturate cleanly here where full-length ones plateaued at 84-85%, that confirms length dilution as the mechanism.

In [8]:
sweep_rows = []
for trigger in ["word", "sent"]:
    for method in ["random", "cbs"]:
        for rate in SWEEP_RATES:
            sweep_rows.append(run_one(method, trigger, rate, seed=42))

sweep_df = pd.DataFrame(sweep_rows)
sweep_pivot = sweep_df.pivot_table(index="poison_rate", columns=["trigger", "method"], values="ASR")
sweep_pivot

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/9880 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Step,Training Loss
500,0.332983
1000,0.200489


...

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

{'CACC': 0.8906666666666667, 'ASR': 0.184, 'ASR_negctrl': 0.14533333333333334, 'method': 'cbs', 'trigger': 'sent', 'poison_rate': 0.02, 'seed': 42, 'n_poisoned': 197}


trigger          sent                word          
method            cbs    random       cbs    random
poison_rate                                        
0.0002       0.102667  0.098667  0.117333  0.116000
0.0005       0.110667  0.100000  0.116000  0.113333
0.0010       0.112000  0.118667  0.118667  0.132000
0.0020       0.118667  0.120000  0.130667  0.125333
0.0050       0.126667  0.118667  0.136000  0.125333
0.0100       0.148000  0.133333  0.125333  0.116000
0.0200       0.184000  0.690667  0.145333  0.126667

In [9]:
def first_rate_reaching(df, trigger, method, threshold=SATURATION_ASR_THRESHOLD):
    sub = df[(df["trigger"] == trigger) & (df["method"] == method)].sort_values("poison_rate")
    hit = sub[sub["ASR"] >= threshold]
    return hit["poison_rate"].iloc[0] if len(hit) else None

saturation_rows = []
for trigger in ["word", "sent"]:
    for method in ["random", "cbs"]:
        saturation_rows.append({
            "trigger": trigger, "method": method,
            f"first_rate_ASR>={SATURATION_ASR_THRESHOLD}": first_rate_reaching(sweep_df, trigger, method),
        })
saturation_summary_df = pd.DataFrame(saturation_rows)
saturation_summary_df

,trigger,method,first_rate_ASR>=0.9
0,word,random,None
1,word,cbs,None
2,sent,random,None
3,sent,cbs,None


## Save

In [10]:
os.makedirs("./results", exist_ok=True)
with pd.ExcelWriter("./results/imdb_short_validation_results.xlsx", engine="openpyxl") as writer:
    sweep_df.to_excel(writer, sheet_name="sweep", index=False)
    saturation_summary_df.to_excel(writer, sheet_name="saturation_summary", index=False)
print("saved ./results/imdb_short_validation_results.xlsx")

saved ./results/imdb_short_validation_results.xlsx


## How to read this against the full-length IMDB result
- **If saturation now happens** (some rate reaches >=90% ASR, unlike the full-length sweep where nothing did up to 0.1): confirms length dilution is the real mechanism behind the plateau.
- **If it still plateaus below 90% even here**: length alone doesn't fully explain it -- something else about IMDB (vocabulary diversity, review structure, genuinely harder examples) is also contributing, and the plateau finding needs a more careful explanation in the paper.
- Also compare the length-bias check above (CBS vs Random mean selected length) against the full-length result (388 vs 305 words) -- if the gap is much smaller here, that's further evidence the effect scales with the available length range, not a fixed property of CBS itself.